# 98 — Campaign B status dashboard（読み取り専用）

M2 / キュー / M3–M6 / backlog / stale lease を表示する。
**計算状態・lock・lease は変更しない。**


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'pipeline_status.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/pipeline_status.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.pipeline_status import collect_pipeline_status, format_status_text

validate_persistent_root()
STATUS = collect_pipeline_status(PERSIST_ROOT)
print(format_status_text(STATUS))


In [ ]:
# Compact view
print(json.dumps({
    'scanned_at': STATUS.get('scanned_at'),
    'm2': STATUS.get('m2'),
    'queues': STATUS.get('queues'),
    'candidate_states': STATUS.get('candidate_states'),
    'leases': STATUS.get('leases'),
    'selected_packages': (STATUS.get('selected') or {}).get('selected_packages'),
}, indent=2, ensure_ascii=False, default=str))
